# Semantic Change Without a Neural Network

**Computational Text Analysis Pack · Module 22 — Distant Reading and Its Critics**

*Technique:* PPMI vectors and nearest neighbours; how a word's company changes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/11-distant-reading-and-its-critics/22b_semantic_change.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

A word's frequency can stay flat while its **meaning** moves. Frequency counting
cannot see that at all; you need to look at the company a word keeps, and see
whether the company changed.

Hamilton et al. (2016) set the neural benchmark for this with word embeddings.
The survey's point is that **count-based methods remain interpretable baselines**,
and for a humanities argument interpretability is often worth more than accuracy:
with PPMI you can point at the actual context words that produced the result.

## The tooling

**gensim** is the standard Python library for word vectors — `Word2Vec`,
`FastText`, and the alignment procedures diachronic work needs. **scikit-learn**
gives you `TruncatedSVD` for the count-based route we take here. The
count-based lineage before word2vec — HAL, LSA, PPMI — is what the survey means
by distributional semantics, and it long predates neural embeddings.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import math

def context_counts(texts, vocab, window=5):
    """Co-occurrence counts restricted to a shared vocabulary."""
    co = Counter()
    uni = Counter()
    for t in texts:
        toks = [w for w in tokenise(t) if w in vocab]
        for i, w in enumerate(toks):
            uni[w] += 1
            for j in range(max(0, i - window), min(len(toks), i + window + 1)):
                if i != j:
                    co[(w, toks[j])] += 1
    return co, uni

old_texts = df_colonial["text"].tolist()
new_texts = df[df["language"] == "English"]["text"].tolist()

c_old = Counter()
for t in old_texts:
    c_old.update(tokenise(t))
c_new = Counter()
for t in new_texts:
    c_new.update(tokenise(t))

# The shared vocabulary is the only fair basis for comparison: a word absent
# from one snapshot has no neighbourhood there to compare.
shared = {w for w in c_old if c_old[w] >= 25 and c_new[w] >= 25}
print(f"shared vocabulary: {len(shared):,} words present in both snapshots")

In [ ]:
co_old, uni_old = context_counts(old_texts, shared)
co_new, uni_new = context_counts(new_texts, shared)

def ppmi_vector(word, co, uni, total_pairs):
    """Positive pointwise mutual information over the shared vocabulary."""
    vec = {}
    for (a, b), n in co.items():
        if a != word:
            continue
        p_ab = n / total_pairs
        p_a = uni[a] / sum(uni.values())
        p_b = uni[b] / sum(uni.values())
        if p_a and p_b:
            val = math.log2(p_ab / (p_a * p_b))
            if val > 0:
                vec[b] = val
    return vec

T_old = sum(co_old.values())
T_new = sum(co_new.values())
print(f"co-occurrence pairs: {T_old:,} old / {T_new:,} modern")

## What changed around a word

The comparison is the top PPMI neighbours in each snapshot. Where the lists
differ, the word's usage has moved — and because PPMI is a count, you can trace
every neighbour back to concordance lines and read them.

In [ ]:
def neighbours(word, k=12):
    v_old = ppmi_vector(word, co_old, uni_old, T_old)
    v_new = ppmi_vector(word, co_new, uni_new, T_new)
    top_old = sorted(v_old, key=v_old.get, reverse=True)[:k]
    top_new = sorted(v_new, key=v_new.get, reverse=True)[:k]
    return top_old, top_new

for TERM in ["native", "development", "chief"]:
    if TERM not in shared:
        continue
    a, b = neighbours(TERM)
    print(f"\n=== '{TERM}' ===")
    print(f"  pre-1929 company : {', '.join(a)}")
    print(f"  modern company   : {', '.join(b)}")
    kept = set(a) & set(b)
    print(f"  shared neighbours: {sorted(kept) if kept else 'none'}")

In [ ]:
# A change score across the shared vocabulary: cosine distance between the two
# PPMI neighbourhoods. High = the word keeps different company now.
def cosine_change(word):
    v1 = ppmi_vector(word, co_old, uni_old, T_old)
    v2 = ppmi_vector(word, co_new, uni_new, T_new)
    if not v1 or not v2:
        return None
    keys = set(v1) | set(v2)
    a = np.array([v1.get(k, 0.0) for k in keys])
    b = np.array([v2.get(k, 0.0) for k in keys])
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 1 - float(a @ b / denom) if denom else None

candidates = [w for w in shared if c_old[w] >= 60 and c_new[w] >= 60][:400]
scores = [(w, cosine_change(w)) for w in candidates]
scores = [(w, s) for w, s in scores if s is not None]
scores.sort(key=lambda t: -t[1])

print("Words whose company changed MOST:\n")
for w, s in scores[:12]:
    print(f"   {w:<16}{s:.3f}   (old {c_old[w]:>5}x, new {c_new[w]:>5}x)")
print("\nWords whose company changed LEAST — the stable core:\n")
for w, s in scores[-8:]:
    print(f"   {w:<16}{s:.3f}")

## Two cautions that decide whether this is evidence

**Frequency confound.** A rare word has a sparse, noisy neighbourhood, so it
scores as "changed" simply for being rare. This is the same confound module 10
found in dispersion, and it needs the same check.

**Genre confound.** Everything from the previous notebook applies. A word can
keep different company because the century changed, or because books and news
are different things.

In [ ]:
freqs = np.array([min(c_old[w], c_new[w]) for w, _ in scores])
vals = np.array([s for _, s in scores])
r = np.corrcoef(np.log(freqs), vals)[0, 1]
print(f"correlation of change score with log frequency: {r:.3f}")
print()
if r < -0.25:
    print("Strongly negative: rare words dominate the 'most changed' list for")
    print("statistical rather than historical reasons. Compare within frequency")
    print("bands, not across them.")
else:
    print("Weak correlation, so the ranking is not purely a frequency artefact —")
    print("but the genre confound is untouched by this check.")

print("\nWithin-band comparison, which is the defensible version:")
bands = [(60, 150), (150, 400), (400, 5000)]
for lo, hi in bands:
    band = [(w, s) for w, s in scores if lo <= min(c_old[w], c_new[w]) < hi]
    if band:
        top = sorted(band, key=lambda t: -t[1])[:5]
        print(f"   {lo}-{hi}x: " + ", ".join(f"{w} ({s:.2f})" for w, s in top))

### 🔧 Your turn 1

Take the highest-scoring word and read ten KWIC lines from each corpus. Does the change the number claims actually appear?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Often yes, and often in a way the number could not have told you — the shift
turns out to be from one sense to another, or from human subject to
institutional subject. Occasionally the number is an artefact: the modern
neighbours are all from three articles about a single event. That is why this
notebook ends where module 11 insisted every count should, at the concordance.
A semantic-change score is a device for deciding which words are worth reading
closely, not a finding in itself.

</details>